# TROPOMI SRON HDO

## already a column averaged product - gridded data (13.5km x 13.5km) plots



In [1]:
import datetime
import sys
import contextlib
from pathlib import Path
import pandas as pd
import xarray as xr
import numpy as np
from netCDF4 import Dataset
import glob
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap

from scipy.interpolate import griddata
from scipy.spatial import cKDTree


In [2]:
#H2O column conversion

#molec/cm**2

#cm**2 --> kg
#(pressure in Pa)*( area in m**2)/(gravity in m/s**2)
P = 101300 #should this be 101325? 
A = 10**(-4)
g = 9.81 

conv_denom = P*A/g

#molec --> g
#(number of molecules)*(atomic weight in g/mol)/(avogadro number in molec/mol)
AW = 18 #should be 18.015?
AV = 6.02214076*10**23

In [3]:
# Specify date range
start_date = datetime.date(2018, 1, 1)
end_date = datetime.date(2024, 12, 31)

dD_list = []
# Loop through the date range and plot Carbon Monoxide for each day
pathlist = glob.glob("/Volumes/New_5TB/ESA_F4R/tropomi2/*.nc")
for t,file in enumerate(pathlist):
    try:
        # Open the netCDF file
        dataset = Dataset(file, 'r')
        #dataset = Dataset("/Volumes/New_5TB/ESA_F4R/tropomi2/S5P_PAL__L2__HDO__S_20180430T121021_20180430T135151_02825_01_100300_20250307T210003.nc", 'r')
        
        # Read the data from your variables
        latitude = dataset['PRODUCT']['latitude'][0,:,:].flatten()
        longitude = dataset['PRODUCT']['longitude'][0,:,:].flatten()
        dD = dataset['PRODUCT']['deltad'][0,:,:].flatten()
        h2o_in = dataset['PRODUCT']['h2o_column'][0,:,:].flatten()
        flag = dataset['PRODUCT']['qa_value'][0,:,:].flatten()
        time_delta = dataset['PRODUCT']['time'][0].item()
        current_date = datetime.datetime(2010,1,1) + datetime.timedelta(seconds=time_delta)
        
        N = h2o_in
        conv_enum = N*AW/AV
        xvmr = conv_enum/conv_denom
        dD = dD*1000.
    
        filter = np.where(flag>=0.7)[0]
        latitude = latitude[filter]
        longitude = longitude[filter]
        dD_col = dD[filter]
        vmr_col = xvmr[filter] 
        
        # Filter the data for Central Africa only
        latitude_max = 13.0
        latitude_min = -15.0
        longitude_max = 32.0
        longitude_min = 8.0
        
        centralaf = np.where(
            (latitude > latitude_min) & (latitude < latitude_max) & 
            (longitude > longitude_min) &(longitude < longitude_max))[0]
        
        latitude = latitude[centralaf]
        longitude = longitude[centralaf]
        dD_col = dD[centralaf]
        vmr_col = xvmr[centralaf]
        dD_col = dD[centralaf]
        vmr_col = xvmr[centralaf]
    
        # Get TROPOMI pixel size in degrees, pixel is 7 x 7 km, 1 degree is roughly 111.1 km
        cris_pixel_size_deg = 7 / 111.1 
        
        # Get the grid for the interpolated values
        grid_lat, grid_lon = np.mgrid[latitude_min:latitude_max-1:cris_pixel_size_deg, longitude_min:longitude_max-1:cris_pixel_size_deg]
        
        print(grid_lat.shape)
        print(grid_lon.shape)
        
        # Interpolate the data using griddata
        try:
            grid_dD_col = griddata((latitude, longitude), dD_col, (grid_lat, grid_lon), method='linear', rescale=True)
            grid_vmr_col = griddata((latitude, longitude), vmr_col, (grid_lat, grid_lon), method='linear', rescale=True)
        
            # Find the distance to the nearest original point for each point in the interpolated grid
            tree = cKDTree(np.vstack((latitude, longitude)).T)
            dist, _ = tree.query(np.vstack((grid_lat.ravel(), grid_lon.ravel())).T)
        
            # Reshape the distance array to have the same shape as the x_col grid
            dist_grid = dist.reshape(grid_dD_col.shape)
        
            # Mask the interpolated values that are too far from any original point
            max_distance_degrees = 3.0
            grid_dD_col[dist_grid > max_distance_degrees] = np.nan
            grid_vmr_col[dist_grid > max_distance_degrees] = np.nan
            
            dD_xr = xr.Dataset(
                            data_vars=dict(deltaD=(["lat","lon"],grid_dD_col),H2O=(["lat","lon"],grid_vmr_col)),
                            coords=dict(
                                time=current_date,
                                lat=(["lat"], grid_lat[:,0]),
                                lon=(["lon"], grid_lon[0,:])
                            ),
                            attrs=dict(
                                description="deltaD",
                                units="permil",
                            ),
                        ) 
            #print(dD_xr)
            dD_list.append(dD_xr)
            #    print('done for date: ', current_date)
            dataset.close()
        except:
            print('region empty for: ',current_date)
            dataset.close()
    except:
        print('problematic file: ',current_date)
        dataset.close()
dD_out = xr.concat(dD_list,dim='time')
dD_out.to_netcdf('/Users/ellendyer/Documents/GitHub/F4R_data/tropomi_gridded_caf.nc')
print(dD_out)

(429, 366)
(429, 366)
region empty for:  2018-04-30 09:08:56
(429, 366)
(429, 366)


/Users/ellendyer/miniconda3/envs/isotope_env/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/ellendyer/miniconda3/envs/isotope_env/lib/python3.13/site-packages/numpy/_core/_methods.py:136: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


(429, 366)
(429, 366)
(429, 366)
(429, 366)
region empty for:  2018-04-30 14:13:26
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
region empty for:  2018-05-01 13:54:29
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
region empty for:  2018-05-03 09:53:33
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
region empty for:  2018-05-05 09:15:38
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
region empty for:  2018-05-05 14:20:09
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
region empty for:  2018-05-06 14:01:11
(429, 366)
(429, 366)
(429, 366)
(429, 366)
region empty for:  2018-05-07 13:42:12
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
region empty for:  2018-05-09 09:41:16
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
(429, 366)
region empty for:  2018-05-10 09:22:17
(429, 366)


RuntimeError: NetCDF: Not a valid ID

In [ ]:
dD_out['deltaD'].mean('time').plot.pcolormesh(vmax=0,vmin=-200)
plt.show()
